# Python Project for Data Science - Examples

This notebook demonstrates practical examples of data collection methods commonly used in data science projects, including API requests and web scraping.

## 1. Working with APIs

### 1.1 Making Basic API Requests with the `requests` Library

In [ ]:
# Import the requests library
import requests

# Sample API endpoint (public API that doesn't require authentication)
url = "https://api.publicapis.org/entries"

# Send a GET request
response = requests.get(url)

# Check if the request was successful
print(f"Status code: {response.status_code}")

# Convert the response to JSON
if response.status_code == 200:
    data = response.json()
    print(f"Number of APIs available: {data['count']}")
    print(f"Example API: {data['entries'][0]['API']}")
else:
    print("Failed to retrieve data")

### 1.2 Handling API Authentication

In [ ]:
# Example of API request with an API key (replace with your actual API key if testing)
def get_weather_data(city, api_key="YOUR_API_KEY"):
    """Function to get weather data for a specified city"""
    # Base URL for OpenWeatherMap API
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    
    # Parameters for the API request
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }
    
    # Making the request
    response = requests.get(base_url, params=params)
    
    # If this was a real implementation with a valid API key:
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Example usage (not executed without a valid API key)
print("To use this function: get_weather_data('London', 'your_actual_api_key')")

### 1.3 Using Financial Data APIs with `yfinance`

The `yfinance` library makes it easy to access financial data from Yahoo Finance:

In [ ]:
# Import the yfinance library
# !pip install yfinance  # Uncomment this line if you need to install the library
import yfinance as yf
import pandas as pd

# Define stock ticker and download data
def get_stock_data(ticker="AAPL", period="1mo"):
    """Get historical stock data for the specified ticker and period"""
    # Create a Ticker object
    stock = yf.Ticker(ticker)
    
    # Get historical market data
    hist = stock.history(period=period)
    
    # Display the first few rows of the data
    return hist.head()

# Example of usage (commented out to avoid execution unless needed)
# apple_data = get_stock_data("AAPL", "1mo")
# apple_data

## 2. Web Scraping Techniques

### 2.1 Basic Web Scraping with BeautifulSoup

In [ ]:
# Import necessary libraries
# !pip install beautifulsoup4 lxml  # Uncomment this line if you need to install the libraries
import requests
from bs4 import BeautifulSoup

def scrape_simple_webpage(url):
    """Function to scrape a simple webpage and extract specific elements"""
    try:
        # Send HTTP request
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Example: Extract the title of the webpage
        title = soup.title.text if soup.title else "No title found"
        
        # Example: Extract all paragraph texts
        paragraphs = [p.text for p in soup.find_all('p')[:3]]  # Limit to first 3 paragraphs
        
        # Return the extracted data
        return {
            "title": title,
            "paragraphs": paragraphs
        }
        
    except requests.exceptions.RequestException as e:
        return {"error": str(e)}

# Example usage (with a simple website that allows scraping)
# Example: scrape_simple_webpage("http://quotes.toscrape.com/")

### 2.2 Extracting Tabular Data from Websites

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests

def scrape_table_data(url, table_index=0):
    """Function to scrape tabular data from a webpage"""
    try:
        # Send HTTP request
        response = requests.get(url)
        response.raise_for_status()
        
        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all tables in the page
        tables = soup.find_all('table')
        
        if not tables or table_index >= len(tables):
            return "No tables found or table_index out of range"
        
        # Get the specified table
        target_table = tables[table_index]
        
        # Extract table headers
        headers = []
        header_row = target_table.find('thead')
        if header_row:
            headers = [th.text.strip() for th in header_row.find_all('th')]
        
        if not headers:  # Try to get headers from first row if thead not found
            first_row = target_table.find('tr')
            if first_row:
                headers = [th.text.strip() for th in first_row.find_all(['th', 'td'])]
        
        # Extract table data
        data = []
        rows = target_table.find_all('tr')
        
        # Skip the first row if it was used for headers
        start_idx = 1 if (not header_row and len(rows) > 0) else 0
        
        for row in rows[start_idx:]:
            row_data = [td.text.strip() for td in row.find_all(['td', 'th'])]
            if row_data:  # Only add non-empty rows
                data.append(row_data)
        
        # Create DataFrame
        if headers and len(headers) == len(data[0]) if data else 0:
            df = pd.DataFrame(data, columns=headers)
        else:
            df = pd.DataFrame(data)
            
        return df
        
    except requests.exceptions.RequestException as e:
        return f"Error: {e}"

# Example usage (this is commented to avoid execution)
# population_data = scrape_table_data(
#     "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population", 
#     table_index=0
# )

### 2.3 Using pandas read_html to Extract Tables

In [ ]:
import pandas as pd

def extract_tables_with_pandas(url, match=None):
    """Extract all tables from a webpage using pandas read_html"""
    try:
        # Read tables directly with pandas
        if match:
            tables = pd.read_html(url, match=match)
        else:
            tables = pd.read_html(url)
            
        print(f"Number of tables found: {len(tables)}")
        
        if len(tables) > 0:
            # Return first table as example
            return tables[0].head()
        else:
            return "No tables found"
            
    except Exception as e:
        return f"Error: {e}"

# Example usage (commented to avoid execution)
# population_data = extract_tables_with_pandas(
#     "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population",
#     match="Country or dependency"
# )

## 3. Data Visualization Examples

### 3.1 Plotting Stock Data with Plotly

In [ ]:
# !pip install plotly  # Uncomment if you need to install plotly
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_stock_data(ticker="AAPL", period="1y"):
    """Create an interactive plot of stock data"""
    # Download stock data
    data = yf.download(ticker, period=period)
    
    # Reset index to make Date a column
    data = data.reset_index()
    
    # Create subplots with shared x-axis
    fig = make_subplots(rows=2, cols=1, 
                         shared_xaxes=True,
                         subplot_titles=(f"{ticker} Stock Price", f"{ticker} Trading Volume"),
                         vertical_spacing=0.1,
                         row_heights=[0.7, 0.3])
    
    # Add price trace
    fig.add_trace(
        go.Scatter(x=data['Date'], y=data['Close'], name="Close Price"),
        row=1, col=1
    )
    
    # Add volume trace
    fig.add_trace(
        go.Bar(x=data['Date'], y=data['Volume'], name="Volume"),
        row=2, col=1
    )
    
    # Update layout
    fig.update_layout(
        title=f"{ticker} Stock Analysis",
        height=800,
        showlegend=True,
        xaxis2_title="Date",
        yaxis_title="Price (USD)",
        yaxis2_title="Volume"
    )
    
    # Display figure
    return fig

# Example usage (commented to avoid execution)
# fig = plot_stock_data("TSLA", "2y")
# fig.show()

## 4. Practical Example Project: Stock Data Analysis

Here's an example of what a complete data science project using API data might look like. This combines data collection, processing, and visualization into a cohesive workflow:

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

class StockAnalyzer:
    def __init__(self, ticker, start_date=None, end_date=None):
        """Initialize the StockAnalyzer with a ticker symbol and date range"""
        self.ticker = ticker
        
        # Set default dates if not provided
        if end_date is None:
            self.end_date = datetime.now().strftime("%Y-%m-%d")
        else:
            self.end_date = end_date
            
        if start_date is None:
            # Default to 1 year ago
            start = datetime.now() - timedelta(days=365)
            self.start_date = start.strftime("%Y-%m-%d")
        else:
            self.start_date = start_date
        
        self.data = None
        self.company_info = None
        
    def fetch_data(self):
        """Download historical stock data"""
        try:
            # Create Ticker object
            stock = yf.Ticker(self.ticker)
            
            # Get company information
            self.company_info = stock.info
            
            # Download historical data
            self.data = stock.history(start=self.start_date, end=self.end_date)
            
            # Reset index to make Date a column
            self.data = self.data.reset_index()
            
            return True
        except Exception as e:
            print(f"Error fetching data: {e}")
            return False
    
    def calculate_metrics(self):
        """Calculate technical indicators and metrics"""
        if self.data is None or len(self.data) == 0:
            print("No data available. Please fetch data first.")
            return False
        
        # Add moving averages
        self.data['MA20'] = self.data['Close'].rolling(window=20).mean()
        self.data['MA50'] = self.data['Close'].rolling(window=50).mean()
        
        # Calculate daily returns
        self.data['Daily_Return'] = self.data['Close'].pct_change() * 100
        
        # Calculate volatility (standard deviation of returns over 20-day window)
        self.data['Volatility'] = self.data['Daily_Return'].rolling(window=20).std()
        
        return True
    
    def generate_dashboard(self):
        """Create an interactive dashboard with multiple plots"""
        if self.data is None or len(self.data) == 0:
            print("No data available. Please fetch data first.")
            return None
        
        # Create figure with subplots
        fig = make_subplots(
            rows=3, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.1,
            subplot_titles=(
                f"{self.ticker} Stock Price",
                "Daily Returns (%)",
                "Trading Volume"
            ),
            row_heights=[0.5, 0.25, 0.25]
        )
        
        # Add price trace with moving averages
        fig.add_trace(
            go.Scatter(x=self.data['Date'], y=self.data['Close'], name="Close Price"),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(x=self.data['Date'], y=self.data['MA20'], name="20-day MA",
                      line=dict(color='orange', width=1)),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(x=self.data['Date'], y=self.data['MA50'], name="50-day MA",
                      line=dict(color='red', width=1)),
            row=1, col=1
        )
        
        # Add daily returns
        fig.add_trace(
            go.Bar(
                x=self.data['Date'], 
                y=self.data['Daily_Return'],
                name="Daily Return",
                marker_color=np.where(self.data['Daily_Return'] >= 0, 'green', 'red')
            ),
            row=2, col=1
        )
        
        # Add volume
        fig.add_trace(
            go.Bar(x=self.data['Date'], y=self.data['Volume'], name="Volume",
                  marker_color='lightblue'),
            row=3, col=1
        )
        
        # Update layout
        company_name = self.company_info.get('shortName', self.ticker) if self.company_info else self.ticker
        
        fig.update_layout(
            title=f"{company_name} ({self.ticker}) Stock Analysis",
            height=900,
            template="plotly_white",
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02),
            xaxis3_title="Date",
            yaxis_title="Price (USD)",
            yaxis2_title="Return (%)",
            yaxis3_title="Volume"
        )
        
        return fig
    
    def get_summary_stats(self):
        """Return summary statistics for the stock"""
        if self.data is None or len(self.data) == 0:
            return "No data available"
        
        # Calculate key statistics
        current_price = self.data['Close'].iloc[-1]
        price_change = self.data['Close'].iloc[-1] - self.data['Close'].iloc[0]
        percent_change = (price_change / self.data['Close'].iloc[0]) * 100
        high = self.data['High'].max()
        low = self.data['Low'].min()
        avg_volume = self.data['Volume'].mean()
        
        summary = {
            "ticker": self.ticker,
            "period": f"{self.start_date} to {self.end_date}",
            "current_price": current_price,
            "price_change": price_change,
            "percent_change": percent_change,
            "high": high,
            "low": low,
            "average_volume": avg_volume,
            "volatility": self.data['Volatility'].iloc[-1] if 'Volatility' in self.data else None
        }
        
        return summary

# Example usage of the StockAnalyzer class (commented to avoid execution)
'''
# Create analyzer for Tesla stock
analyzer = StockAnalyzer("TSLA")

# Fetch and process data
if analyzer.fetch_data():
    analyzer.calculate_metrics()
    
    # Get summary statistics
    summary = analyzer.get_summary_stats()
    for key, value in summary.items():
        print(f"{key}: {value}")
        
    # Create and display dashboard
    fig = analyzer.generate_dashboard()
    fig.show()
'''

## 5. Error Handling and Best Practices

### 5.1 Handling Rate Limits and Timeouts

In [ ]:
import requests
import time

def rate_limited_request(url, max_retries=3, retry_delay=5):
    """Make a request with retry logic for rate limits and server errors"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    for attempt in range(max_retries):
        try:
            # Make the request with a timeout
            response = requests.get(url, headers=headers, timeout=10)
            
            # Check for rate limiting response codes
            if response.status_code == 429:  # Too Many Requests
                print(f"Rate limited. Waiting {retry_delay} seconds before retry {attempt+1}/{max_retries}")
                time.sleep(retry_delay)
                # Exponential backoff for successive retries
                retry_delay *= 2
                continue
                
            # Check for server errors (5xx)
            elif 500 <= response.status_code < 600:
                print(f"Server error: {response.status_code}. Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                continue
                
            # Return successful response
            return response
            
        except requests.exceptions.Timeout:
            print(f"Request timed out. Retrying {attempt+1}/{max_retries}...")
            continue
            
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                continue
            else:
                raise
                
    # If we've exhausted our retries
    raise Exception(f"Failed to get response after {max_retries} attempts")

# Example usage (commented to avoid execution)
# try:
#     response = rate_limited_request("https://api.example.com/data")
#     print("Success!")
# except Exception as e:
#     print(f"Failed: {e}")

### 5.2 Storing Collected Data

In [ ]:
import pandas as pd
import json
import os
from datetime import datetime

def save_data(data, filename=None, format="csv", folder="data"):
    """Save data to disk in various formats"""
    # Create folder if it doesn't exist
    if not os.path.exists(folder):
        os.makedirs(folder)
        
    # Generate filename with timestamp if not provided
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"data_{timestamp}"
    
    # Full path with appropriate extension
    if not filename.endswith(f".{format}"):
        filepath = os.path.join(folder, f"{filename}.{format}")
    else:
        filepath = os.path.join(folder, filename)
    
    try:
        # Save based on format
        if format.lower() == "csv":
            # Convert to DataFrame if it's not already
            if not isinstance(data, pd.DataFrame):
                data = pd.DataFrame(data)
            data.to_csv(filepath, index=False)
            
        elif format.lower() == "json":
            # If it's a DataFrame, convert to dict
            if isinstance(data, pd.DataFrame):
                data = data.to_dict(orient="records")
                
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
                
        elif format.lower() == "excel" or format.lower() == "xlsx":
            # Ensure extension is .xlsx
            if not filepath.endswith(".xlsx"):
                filepath = filepath.rsplit(".", 1)[0] + ".xlsx"
                
            # Convert to DataFrame if it's not already
            if not isinstance(data, pd.DataFrame):
                data = pd.DataFrame(data)
            data.to_excel(filepath, index=False)
            
        else:
            return f"Unsupported format: {format}"
            
        return f"Data successfully saved to {filepath}"
        
    except Exception as e:
        return f"Error saving data: {e}"

# Example usage (commented to avoid execution)
# sample_data = pd.DataFrame({
#     'Name': ['Alice', 'Bob', 'Charlie'],
#     'Age': [25, 30, 35],
#     'City': ['New York', 'London', 'Paris']
# })
# result = save_data(sample_data, filename="sample_data", format="csv")
# print(result)

## Conclusion

This notebook has demonstrated various techniques for collecting and visualizing data for data science projects. These examples serve as a foundation that you can build upon for your own specific projects. Remember to always consider ethical aspects of data collection, especially when web scraping, and to properly handle errors and rate limits when working with APIs.